# Customer Review Sentiment Analysis for an E-commerce Store

**Client goal:** automatically label product reviews as **Positive, Neutral or Negative**, and see
what unhappy customers complain about, instead of reading thousands of reviews by hand.

**Data:** Women's E-Commerce Clothing Reviews, about 23k real customer reviews with 1-5 star ratings
(Kaggle `nicapotato/womens-ecommerce-clothing-reviews`).

**Plan:**
1. Clean the text, remove empty and duplicate reviews
2. Create labels from the star rating: 1-2 = Negative, 3 = Neutral, 4-5 = Positive
3. TF-IDF features inside a pipeline (no leakage)
4. Compare Naive Bayes, Logistic Regression and Linear SVM with stratified cross-validation
5. Evaluate on a held-out test set (accuracy, precision, recall, F1, confusion matrix)
6. Save the best model for the Streamlit app

In [ ]:
import os, subprocess, sys
if os.path.exists("/kaggle"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn==1.7.2"], check=False)

In [ ]:
%%writefile text_utils.py
"""Text helpers shared by the training notebook and the Streamlit app."""
import re

import numpy as np
import pandas as pd

LABELS = ["Negative", "Neutral", "Positive"]


def rating_to_label(r: int) -> str:
    return "Negative" if r <= 2 else ("Neutral" if r == 3 else "Positive")


def clean_text(s) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r"https?://\S+", " ", s)
    s = s.replace("&#39;", "'").replace("&amp;", "&")
    return re.sub(r"\s+", " ", s).strip()


def combine(title, text) -> str:
    title, text = clean_text(title), clean_text(text)
    return f"{title}. {text}" if title else text


def class_weights(model):
    """Per-class word weights of the final classifier (rows follow model.classes_)."""
    clf = model[-1]
    if hasattr(clf, "coef_"):
        return np.asarray(clf.coef_)
    if hasattr(clf, "calibrated_classifiers_"):  # CalibratedClassifierCV(LinearSVC)
        return np.mean([c.estimator.coef_ for c in clf.calibrated_classifiers_], axis=0)
    return np.asarray(clf.feature_log_prob_)  # Naive Bayes


def explain(model, text: str, label: str, top: int = 8) -> pd.DataFrame:
    """Words/phrases in `text` that pushed the prediction towards `label` (or away from it)."""
    vec = model[0]
    x = vec.transform([text])
    W = class_weights(model)
    k = list(model.classes_).index(label)
    others = np.delete(W, k, axis=0).mean(axis=0)
    idx = x.indices
    contrib = x.data * (W[k, idx] - others[idx])
    terms = vec.get_feature_names_out()[idx]
    df = pd.DataFrame({"term": terms, "contribution": contrib})
    return df.reindex(df["contribution"].abs().sort_values(ascending=False).index).head(top)


def top_terms(model, label: str, n: int = 15) -> list[str]:
    W = class_weights(model)
    k = list(model.classes_).index(label)
    score = W[k] - np.delete(W, k, axis=0).mean(axis=0)
    return list(model[0].get_feature_names_out()[np.argsort(score)[::-1][:n]])

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from text_utils import LABELS, combine, explain, rating_to_label, top_terms

SEED = 42
for d in ["data", "model", "reports"]:
    Path(d).mkdir(exist_ok=True)
print("scikit-learn", sklearn.__version__)


def find(name):
    for base in [Path("/kaggle/input"), Path("data")]:
        hits = list(base.rglob(name)) if base.exists() else []
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{name} not found; attach the dataset or put it in data/")

## 1. Load and clean

In [ ]:
raw = pd.read_csv(find("Womens Clothing E-Commerce Reviews.csv"), index_col=0)
print(raw.shape)
raw.head(3)

In [ ]:
df = raw.copy()
info = {"rows_raw": len(df), "missing_review_text": int(df["Review Text"].isna().sum())}
df = df.dropna(subset=["Review Text", "Rating"])
df["text"] = [combine(t, r) for t, r in zip(df["Title"], df["Review Text"])]
df = df[df["text"].str.len() >= 10]
info["duplicate_texts"] = int(df.duplicated("text").sum())
df = df.drop_duplicates("text")  # the same text in train and test would inflate the scores
df["sentiment"] = df["Rating"].astype(int).map(rating_to_label)
info["rows_clean"] = len(df)
info["class_counts"] = df["sentiment"].value_counts().reindex(LABELS).to_dict()
print(json.dumps(info, indent=2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
df["Rating"].value_counts().sort_index().plot.bar(ax=axes[0], rot=0, title="Star ratings")
df["sentiment"].value_counts().reindex(LABELS).plot.bar(ax=axes[1], rot=0, color=["#d62728", "#999999", "#2ca02c"],
                                                         title="Sentiment labels (imbalanced)")
(pd.crosstab(df["Department Name"], df["sentiment"], normalize="index")[LABELS]
 .plot.barh(stacked=True, ax=axes[2], color=["#d62728", "#999999", "#2ca02c"],
            title="Sentiment share by department"))
axes[2].legend(fontsize=7)
fig.tight_layout()
fig.savefig("reports/eda_sentiment.png", dpi=110)
plt.show()

About three quarters of reviews are positive, so **accuracy alone is misleading** (always saying
"Positive" would already score about 77%). Models are compared on **macro F1**, which gives the
Negative and Neutral classes equal weight, and the classifiers use balanced class weights.

## 2. Train / test split and model comparison
TF-IDF keeps stop-words such as "not" and "but", because they carry sentiment ("not flattering").

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(df["text"], df["sentiment"], test_size=0.2,
                                          stratify=df["sentiment"], random_state=SEED)


def tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True,
                           strip_accents="unicode")


def build_models():
    return {
        "Naive Bayes": Pipeline([("tfidf", tfidf()), ("clf", ComplementNB(alpha=0.3))]),
        "Logistic Regression": Pipeline([("tfidf", tfidf()), ("clf", LogisticRegression(
            C=4, class_weight="balanced", max_iter=3000, random_state=SEED))]),
        "Linear SVM": Pipeline([("tfidf", tfidf()), ("clf", CalibratedClassifierCV(
            LinearSVC(C=0.3, class_weight="balanced", random_state=SEED), cv=3))]),
    }


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rows, fitted, cms = [], {}, {}
for name, model in build_models().items():
    res = cross_validate(model, X_tr, y_tr, cv=cv, scoring=["f1_macro", "accuracy"], n_jobs=4)
    model.fit(X_tr, y_tr)
    p = model.predict(X_te)
    rows.append({"model": name, "cv_f1_macro_mean": res["test_f1_macro"].mean(),
                 "cv_f1_macro_std": res["test_f1_macro"].std(),
                 "accuracy": accuracy_score(y_te, p),
                 "precision_macro": precision_score(y_te, p, average="macro"),
                 "recall_macro": recall_score(y_te, p, average="macro"),
                 "f1_macro": f1_score(y_te, p, average="macro"),
                 "negative_recall": recall_score(y_te, p, labels=["Negative"], average="macro")})
    fitted[name] = model
    cms[name] = confusion_matrix(y_te, p, labels=LABELS).tolist()
results = pd.DataFrame(rows).round(4)
best = results.sort_values("cv_f1_macro_mean", ascending=False).iloc[0]["model"]
best_model = fitted[best]
results.to_csv("reports/model_comparison.csv", index=False)
print("Selected (best CV macro F1):", best)
results

## 3. Evaluation on the held-out test set

In [ ]:
per_class = pd.DataFrame(classification_report(y_te, best_model.predict(X_te), output_dict=True)).T.round(4)
per_class.to_csv("reports/per_class_report.csv")
per_class

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (name, cm) in zip(axes, cms.items()):
    ConfusionMatrixDisplay(np.array(cm), display_labels=LABELS).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name + (" (selected)" if name == best else ""))
fig.tight_layout()
fig.savefig("reports/confusion_matrices.png", dpi=110)
plt.show()

ax = results.set_index("model")[["accuracy", "precision_macro", "recall_macro", "f1_macro"]].plot.bar(
    figsize=(9, 4.5), rot=0, ylim=(0.4, 1.0))
ax.set_title("Held-out test performance (macro averages over 3 classes)")
ax.legend(loc="upper right", ncol=4, fontsize=8)
plt.tight_layout()
plt.savefig("reports/model_comparison.png", dpi=120)
plt.show()

## 4. What drives each sentiment?

In [ ]:
drivers = {lab: top_terms(best_model, lab, 15) for lab in LABELS}
for lab, terms in drivers.items():
    print(f"{lab}: {', '.join(terms)}")

example = "The fabric is cheap and the fit is not flattering at all, returning it."
print("\nExample:", example, "->", best_model.predict([example])[0])
explain(best_model, example, best_model.predict([example])[0])

## 5. Save model and outputs for the app
The selected pipeline (TF-IDF + classifier) is refitted on all labelled reviews before saving.

In [ ]:
final = build_models()[best].fit(df["text"], df["sentiment"])
joblib.dump(final, "model/best_model.joblib", compress=3)

sample = df.loc[X_te.index].groupby("sentiment").sample(n=60, random_state=SEED)
sample[["Clothing ID", "Department Name", "Class Name", "Title", "Review Text", "Rating"]] \
    .sample(frac=1, random_state=SEED).to_csv("data/sample_reviews.csv", index=False)

meta = {"selected_model": best, "selection_rule": "highest mean 5-fold CV macro F1 on the training set",
        "sklearn_version": sklearn.__version__, "clean_info": info,
        "train_size": len(X_tr), "test_size": len(X_te),
        "results": results.to_dict(orient="records"), "confusion_matrices": cms,
        "label_rule": "1-2 stars = Negative, 3 = Neutral, 4-5 = Positive", "drivers": drivers}
Path("model/metrics.json").write_text(json.dumps(meta, indent=2))


def md_table(df, index=False):
    df = df.reset_index() if index else df
    head = "| " + " | ".join(map(str, df.columns)) + " |\n|" + "---|" * len(df.columns) + "\n"
    return head + "\n".join("| " + " | ".join(str(v) for v in row) + " |" for row in df.values)


r = results.set_index("model").loc[best]
report = f"""# Evaluation Report: Review Sentiment Analysis

## Data
- Women's E-Commerce Clothing Reviews: {info['rows_raw']:,} reviews; {info['missing_review_text']:,} without
  text and {info['duplicate_texts']} duplicate texts removed -> {info['rows_clean']:,} reviews.
- Labels from stars ({meta['label_rule']}): {info['class_counts']}.
- Stratified 80/20 split ({len(X_tr):,} train / {len(X_te):,} test).

## Method
- Text = title + review, cleaned; TF-IDF on words and 2-word phrases (fitted inside each CV fold).
- Models: Complement Naive Bayes, Logistic Regression and Linear SVM (both with balanced class weights).
- 5-fold stratified CV on the training set; best mean **macro F1** is selected (accuracy is misleading
  because 77% of reviews are positive).

## Results
{md_table(results)}

**Selected model: {best}**: test macro F1 {r['f1_macro']:.4f}, accuracy {r['accuracy']:.4f},
recall on Negative reviews {r['negative_recall']:.4f}.

### Per-class results (selected model)
{md_table(per_class, index=True)}

![Confusion matrices](confusion_matrices.png)
![Model comparison](model_comparison.png)

### Words that push towards each sentiment
- **Negative:** {', '.join(drivers['Negative'])}
- **Neutral:** {', '.join(drivers['Neutral'])}
- **Positive:** {', '.join(drivers['Positive'])}

## Limitations
- **Neutral** (3-star) reviews are the hardest class: they usually mix praise and complaints, and
  the same text could get 2 or 4 stars from different people. Expect most errors there.
- Labels come from star ratings, which are noisy (some 5-star reviews mention problems).
- Trained on English clothing reviews; other product types or languages (e.g. Arabic) need retraining
  on the client's own reviews. Sarcasm and very short texts ("ok") are often misread.
"""
Path("reports/evaluation_report.md").write_text(report, encoding="utf-8")
print(report)